In [3]:
import os
from huggingface_hub import snapshot_download

model_name = "unsloth/Qwen3-4B-bnb-4bit"  # Correct unsloth 4-bit model repo
save_dir = os.path.join(os.getcwd(), "models", "Qwen3-4B-unsloth-4bit")

os.makedirs(save_dir, exist_ok=True)
print(f"Model will be downloaded: {save_dir}")

if os.path.exists(save_dir) and os.listdir(save_dir):
    print("Model already exists, skipping installation.")
else:
    snapshot_download(
        repo_id=model_name,
        local_dir=save_dir,
        local_dir_use_symlinks=False
    )
    print("\nDownloaded successfully!")

print(f"Files: {os.listdir(save_dir)}")

Model will be downloaded: /home/yesoytur/APilus/models/Qwen3-4B-unsloth-4bit
Model already exists, skipping installation.
Files: ['.cache', 'chat_template.jinja', 'generation_config.json', 'config.json', '.gitattributes', 'vocab.json', 'added_tokens.json', 'special_tokens_map.json', 'model.safetensors', 'tokenizer_config.json', 'merges.txt', 'tokenizer.json', 'README.md']


In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, GenerationConfig
import os
import time

model_path = os.path.join(os.getcwd(), "models", "Qwen3-4B-unsloth-4bit")

tokenizer = AutoTokenizer.from_pretrained(model_path)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto"
)

print("✅ Model loaded!")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

✅ Model loaded!


In [5]:
def ask(
    prompt,
    enable_thinking=False,
    max_new_tokens=512,
    temperature=0.7,
    top_p=0.9,
    top_k=50,
    repetition_penalty=1.1,
    do_sample=True,
    seed=None,
    verbose=True,
):
    if seed is not None:
        import torch
        torch.manual_seed(seed)

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    input_token_count = model_inputs.input_ids.shape[1]

    gen_config = GenerationConfig(
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature if do_sample else None,
        top_p=top_p if do_sample else None,
        top_k=top_k if do_sample else None,
        repetition_penalty=repetition_penalty,
        pad_token_id=tokenizer.eos_token_id,
    )

    start_time = time.perf_counter()
    generated_ids = model.generate(**model_inputs, generation_config=gen_config)
    elapsed = time.perf_counter() - start_time

    output_ids = generated_ids[0][input_token_count:]
    output_token_count = len(output_ids)
    tokens_per_second = output_token_count / elapsed

    response = tokenizer.decode(output_ids, skip_special_tokens=True).strip()

    if verbose:
        print(f"\n{'─'*48}")
        print(f"  🌡  Temperature         : {temperature}")
        print(f"  🎯 Top-p               : {top_p}")
        print(f"  🔝 Top-k               : {top_k}")
        print(f"  🔁 Repetition penalty  : {repetition_penalty}")
        print(f"  🎲 Sampling            : {do_sample}  |  Seed: {seed}")
        print(f"  {'─'*42}")
        print(f"  📥 Input tokens        : {input_token_count:>6} tokens")
        print(f"  📤 Output tokens       : {output_token_count:>6} tokens")
        print(f"  ⏱  Generation time     : {elapsed:>6.2f} s")
        print(f"  ⚡ Tokens per second   : {tokens_per_second:>6.1f} tok/s")
        print(f"{'─'*48}\n")

    return response

print("✅ ask() ready!")

✅ ask() ready!


In [6]:
print(ask("Tell me a joke"))

Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



────────────────────────────────────────────────
  🌡  Temperature         : 0.7
  🎯 Top-p               : 0.9
  🔝 Top-k               : 50
  🔁 Repetition penalty  : 1.1
  🎲 Sampling            : True  |  Seed: None
  ──────────────────────────────────────────
  📥 Input tokens        :     16 tokens
  📤 Output tokens       :     20 tokens
  ⏱  Generation time     :   0.89 s
  ⚡ Tokens per second   :   22.5 tok/s
────────────────────────────────────────────────

Why don't skeletons fight each other?  
Because they don't have the guts! 😄


In [10]:
print(ask("Compare TCP and UDP", temperature=0.8, top_p=0.95))

Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



────────────────────────────────────────────────
  🌡  Temperature         : 0.8
  🎯 Top-p               : 0.95
  🔝 Top-k               : 50
  🔁 Repetition penalty  : 1.1
  🎲 Sampling            : True  |  Seed: None
  ──────────────────────────────────────────
  📥 Input tokens        :     16 tokens
  📤 Output tokens       :    512 tokens
  ⏱  Generation time     :  12.20 s
  ⚡ Tokens per second   :   42.0 tok/s
────────────────────────────────────────────────

TCP (Transmission Control Protocol) and UDP (User Datagram Protocol) are both Internet protocols used for transmitting data over a network, but they differ significantly in terms of reliability, speed, and usage. Here's a detailed comparison between TCP and UDP:

---

### **1. Connection-Oriented vs. Connectionless**
- **TCP**: Connection-oriented  
  - Establishes a connection before sending data.
  - Uses a three-way handshake (SYN, SYN/ACK, ACK) to ensure both ends are ready.

- **UDP**: Connectionless  
  - No prior connect

In [19]:
print(ask("Acıbadem Üniversitesi Bilgisayar Mühendisliği başkanı 'Ahmet Bulut' aklında tut.", top_p=0.95, temperature=0.7))

Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



────────────────────────────────────────────────
  🌡  Temperature         : 0.7
  🎯 Top-p               : 0.95
  🔝 Top-k               : 50
  🔁 Repetition penalty  : 1.1
  🎲 Sampling            : True  |  Seed: None
  ──────────────────────────────────────────
  📥 Input tokens        :     39 tokens
  📤 Output tokens       :    328 tokens
  ⏱  Generation time     :   7.92 s
  ⚡ Tokens per second   :   41.4 tok/s
────────────────────────────────────────────────

Elbette! Acıbadem Üniversitesi Bilgisayar Mühendisliği Bölümü Başkanı Ahmet Bulut’un aklında tutulması, bu kişiyi daha iyi anlayabilmek için bazı bilgileri birleştirmek istiyorsunuz mu? Örneğin:

- **Kariyeri**: Ahmet Bulut, Acıbadem Üniversitesinde Bilgisayar Mühendisliği Bölümü Başkannya terfi etti. Bu konumda akademik ve yönetim alanında deneyime sahiptir.
- **İlgi Alanları**: Bilgisayar Mühendisliği, yapay zeka, veri bilimi, yazılım mühendisliği gibi alanlarla ilgilidir.
- **Yönetim Görevleri**: Bölüm Başkanı olarak, progra

In [17]:
print(ask("Acıbadem Üniversitesi bölüm başkanı kimdir ?"))

Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



────────────────────────────────────────────────
  🌡  Temperature         : 0.7
  🎯 Top-p               : 0.9
  🔝 Top-k               : 50
  🔁 Repetition penalty  : 1.1
  🎲 Sampling            : True  |  Seed: None
  ──────────────────────────────────────────
  📥 Input tokens        :     24 tokens
  📤 Output tokens       :    366 tokens
  ⏱  Generation time     :   8.56 s
  ⚡ Tokens per second   :   42.8 tok/s
────────────────────────────────────────────────

Acıbadem Üniversitesi'nin bölüm başkanları, üniversitenin farklı fakültelerindeki bölümlerin yönetimi görevini yapan kişilerdir. Ancak "bölüm başkanı" olarak genellikle bir fakülte içindeki bir bölümün (örneğin; Matematik Bölümü, Fizik Bölümü vb.) başkanını ifade eder.

Ancak sadece "Acıbadem Üniversitesi bölüm başkanı kimdir?" sorusu, üniversiteyi tüm olarak değil, belli bir fakülte veya bölümün başkanını istiyor olabilir. Bu nedenle, daha net bir cevap vermek için hangi fakülte ya da bölüme dair bilgi verilmesi gerekir.

Ancak